# Chapitre 2 : Working with text data
Notebook d'accompagnement pour l'exécution des extraits de code du chapitre 2 de *Build a Large Language Model (from scratch)*.

In [1]:
# Importation de base (typiquement PyTorch pour ce livre)
# Assure-toi que PyTorch est installé dans ton environnement : pip install torch
import torch

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.7.1+cpu


## 2.2 Tokenizing text
Téléchargement du texte d'exemple ("The Verdict") utilisé pour l'entraînement.

In [2]:
# Télécharger le fichier "the-verdict.txt" depuis l'url ci-dessous.
# url: "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
# Ensuite, lire le fichier avec `open(...)` et afficher le nombre total de caractères, puis les 99 premiers caractères.
import urllib.request
url = ("https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt")
file_path = "the-verdict.txt"
urllib.request.urlretrieve(url, file_path)

('the-verdict.txt', <http.client.HTTPMessage at 0x1ed6782da90>)

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


Application des **expressions régulières** (regex) avec Python pour diviser le texte en mots et ponctuation.

In [4]:
import re

# Utiliser re.split avec les expressions vues pour tester la séparation sur ce texte :
text = "Hello, world. This, is a test."

# 1. Séparation basique (espaces) : `r'(\s)'`
# 2. Séparation modifiée (espaces, virgules et points) : `r'([,.]|\s)'`
# 3. Retirer les espaces avec `strip()`

In [5]:
result = re.split(r'(\s)', text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


In [6]:
result = re.split(r'([,.]|\s)', text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


In [7]:
result = [item for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


Modification finale pour prendre en compte d'autres signes de ponctuation et les double-tirets, avec application sur le texte de *The Verdict*.

In [8]:
# Utiliser la regex finale : r'([,.:;?_!"()\']|--|\s)'
# 1. Tester sur `text2 = "Hello, world. Is this-- a test?"`
# 2. Appliquer au texte complet du livre (`raw_text`) et afficher: 
#    - Le nombre total de tokens
#    - Les 30 premiers tokens

In [9]:
text = "Hello, world. Is this-- a test?"
result = re.split(r'([,.:;?_!]|--|\s)', text)
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


In [10]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed))

4690


In [11]:
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## 2.3 Converting tokens into token IDs
Création du vocabulaire à partir des tokens uniques de _The Verdict_.

In [12]:
# 1. Utiliser set() pour garder les tokens uniques de `preprocessed` puis les trier par ordre alphabétique (`sorted()`). Assigner le résultat à `all_words`.
# 2. Récupérer la taille de la liste (`vocab_size`) et l'afficher (résultat attendu: 1130).
# 3. Créer un dictionnaire `vocab` associant chaque token à un nombre de 0 à vocab_size-1. (utiliser enumerate() permet de le faire facilement)
# 4. Afficher les 50 premiers éléments de ce dictionnaire pour s'assurer de sa structure.

In [13]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)
print(vocab_size)

1130


In [14]:
vocab = {token:integer for integer, token in enumerate(all_words)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


### Implémentation d'une classe Tokenizer
Création de la classe `SimpleTokenizerV1` qui gère le codage (`encode`) et décodage (`decode`) des listes de chaînes ou d'IDs.

In [15]:
import re

class SimpleTokenizerV1:
    def __init__(self, vocab):
        # 1. Stocker le dictionnaire vocab dans un argument d'objet `self.str_to_int`
        self.str_to_int = vocab
        # 2. Créer le dictionnaire inverse (mapping: entier vers chaîne) dans `self.int_to_str`
        self.int_to_str = {i:s for s, i in vocab.items()}
    
    def encode(self, text):
        # 1. Appliquer re.split sur le `text` rentré avec l'expression régulière précédente (r'([,.?_!"()\']|--|\s)')
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        # 2. Filtrer et stripper les tokens
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # 3. Retourner la liste d'IDs en parcourant le `self.str_to_int`
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        # 1. Parcourir chaque i dans `ids` pour générer une liste de chaîne depuis le mapping `self.int_to_str`
        # 2. Joindre toutes ces chaînes avec " ".join(...)
        text = " ".join([self.int_to_str[i] for i in ids])
        # 3. Utiliser re.sub(r'\s+([,.?!"()\'])', r'\1', text) pour retirer les blancs avant la ponctuation
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text
    

Test du tokenizer avec des cas valides et un cas d'échec (OOTV - Out of Vocabulary).

In [16]:
# 1. Instancier `SimpleTokenizerV1` avec le dictionnaire `vocab` défini au début.
# 2. Utiliser la méthode `encode` sur la phrase : 
#    """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
# 3. Afficher la liste des IDs en résultat.

tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)


[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [17]:
# 4. Utiliser la méthode `decode` sur la liste générée et imprimer le texte récupéré.
print(tokenizer.decode(ids))

" It' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.


In [18]:
# 5. [TEST EXCEPTION] Assayer d'encoder le texte `"Hello, do you like tea?"` et imprimer. 
#    Un KeyError devrait être levé sur "Hello".
text = "Hello, do you like tea?"
try:
    print(tokenizer.encode(text))
except KeyError as e:
    print(f"KeyError: {e} not in vocabulary")

KeyError: 'Hello' not in vocabulary


## 2.4 Adding special context tokens
Modifier le vocabulaire pour gérer les mots inconnus (`<|unk|>`) et la séparation de textes indépendants (`<|endoftext|>`).

In [19]:
# 1. Utiliser `preprocessed` (les mots uniques de la section précédente).
# 2. Utiliser `extend()` pour ajouter les deux nouveaux tokens spéciaux : "<|endoftext|>" et "<|unk|>" à la liste.
# 3. Créer un nouveau dictionnaire `vocab` via `enumerate()` et afficher sa nouvelle longueur (doit être 1132).
# 4. Boucler sur les 5 derniers éléments du `vocab` et les afficher pour confirmer l'insertion.

In [20]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer, token in enumerate(all_tokens)}

print(len(vocab.items()))

1132


In [21]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


### Création de SimpleTokenizerV2 et tests
Validation du `SimpleTokenizerV2` sur un ensemble concaténé comportant des mots hors "The Verdict".

In [22]:
# 1. Copier la classe `SimpleTokenizerV1` et la nommer `SimpleTokenizerV2`.
# 2. Modifier `encode()` en ajoutant un fallback :
#    Mots existants (in self.str_to_int) = mot
#    Autrement (else) = "<|unk|>"

# --- TESTS ---
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

# 3. Join text1 et text2 avec la chaîne de séparateur " <|endoftext|> " dans un nouvel objet `text`.
# 4. Imprimer `text` et s'assurer de sa forme correcte.
# 5. Instancier SimpleTokenizerV2, encoder `text` puis l'imprimer pour observer les IDs remplaçant "Hello" et "palace" avec (1131).
# 6. Décoder ces IDs pour constater le remplacement effectif du mot initial par "<|unk|>".

In [23]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s, i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        preprocessed = [
            item if item in self.str_to_int else "<|unk|>" for item in preprocessed
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
    
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text 

In [24]:
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = text1 + " <|endoftext|> " + text2
print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [25]:
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]


In [26]:
print(tokenizer.decode(tokenizer.encode(text)))

<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


## 2.5 Byte pair encoding

In [27]:
# 1. Installer tiktoken si ce n'est pas déjà fait (ex: !pip install tiktoken)
# 2. Importer la fonction version depuis importlib.metadata et tiktoken pour vérifier la version 
# 3. Afficher la version de tiktoken

In [28]:
!pip install tiktoken --quiet


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: C:\Users\vleon\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [29]:
from importlib.metadata import version
import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.9.0


In [30]:
# 1. Instancier le tokenizer BPE "gpt2" avec tiktoken
# 2. Encoder le texte "Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace." en autorisant <|endoftext|>
# 3. Afficher la liste des identifiants (integers)
# 4. Décoder ces identifiants pour vérifier la reconstruction du texte

In [31]:
tokenizer = tiktoken.get_encoding("gpt2")

In [32]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


In [33]:
strings = tokenizer.decode(integers)
print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace.


### Exercice 2.1 : Encodage par paire d'octets de mots inconnus

In [34]:
# 1. Tester le tokenizer BPE sur le mot inconnu "Akwirw ier" (encoder lettre par lettre ou syllabe par syllabe pour identifier les IDs)
# 2. Utiliser la fonction de décodage sur la liste d'entiers donnés [33901, 86, 343, 86, 220, 959] pour vérifier qu'elle reproduit le mot d'origine

In [35]:
tokens = ["Ak", "w", "ir", "w", " ", "ier"]
token_ids = []
for token in tokens:
    if token == " ":
        print(f"' ': {tokenizer.encode(" ")}")
    else:
        print(f"{token}: {tokenizer.encode(token)}")
    token_ids.append(tokenizer.encode(token)[0])
print(token_ids)


Ak: [33901]
w: [86]
ir: [343]
w: [86]
' ': [220]
ier: [959]
[33901, 86, 343, 86, 220, 959]


In [36]:
print(tokenizer.decode(token_ids))

Akwirw ier


## 2.6 Data sampling with a sliding window

In [37]:
# Lire le contenu du fichier the-verdict.txt, encoder le texte avec tokenizer.encode() 
# puis afficher la longueur totale du texte encodé

In [38]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


In [39]:
# Retirer les 50 premiers tokens de la liste en assignant le reste à une variable "enc_sample"

In [40]:
enc_sample = enc_text[50:]

In [41]:
# Créer une variable "context_size" égale à 4.
# Extraire les 4 premiers tokens depuis enc_sample pour créer l'entrée "x"
# Extraire les tokens de l'index 1 à 5 pour créer la cible "y"
# Afficher x et y

In [42]:
# The context size determines how many tokens are included in the input
context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]
print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


In [43]:
# Créer une ligne de code pour itérer sur la taille du contexte de 1 jusqu'à context_size inclus.
# Pour chaque itération, extraire le "context" d'entrée jusqu'à l'index courant.
# Extraire le token désiré (cible) correspondant au pas suivant
# Afficher le format list complet du contexte et l'identifiant cible -> "context ----> desired"

In [44]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"{context} ----> {desired}")


[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [45]:
# Refaire la même boucle que précédemment, mais cette fois-ci utiliser
# tokenizer.decode() pour convertir les identifiants (IDs) en texte brut
# avant de les afficher.

In [46]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(f"{tokenizer.decode(context)} ----> {tokenizer.decode([desired])}")

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


In [47]:
# Implémenter la classe GPTDatasetV1
# Cette classe doit hériter de PyTorch Dataset.
# Dans la méthode __init__, tokéniser tout le texte "txt" et créer les listes "input_ids" et "target_ids".
# Utiliser une fenêtre glissante basée sur "max_length" et "stride" pour remplir ces listes.
# Coder la méthode __len__ pour renvoyer le nombre total d'éléments.
# Coder la méthode __getitem__ pour extraire et renvoyer une paire de tenseurs d'entrée et cible.

In [48]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt)

        # Uses a sliding window to chunk the book into overlapping sequences of max_length
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i: i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # Returns the total number of row in the dataset
        return len(self.input_ids)
        
    def __getitem__(self, idx):
        # Returns a single row from the dataset
        return self.input_ids[idx], self.target_ids[idx]


In [49]:
# Implémenter la fonction create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0)
# Cette fonction doit amorcer le tokenizer "gpt2", créer une instance du Dataset avec GPTDatasetV1
# Finalement, elle doit retourner un composant DataLoader de PyTorch contenant le dataset avec tous les paramètres transmis.

In [50]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, 
                         stride=128, shuffle=True, drop_last=True, num_workers=0):
    # Initializes the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")
    # Creates dataset
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        # drop_last = True drops the last batch if it is shorter than the specified batch_size to prevent loss spikes during training.
        drop_last=drop_last,
        # The number of CPU processes to use for preprocessing
        num_workers=num_workers
    )                          

    return dataloader           
                                    

In [51]:
# Test: Avec the-verdict.txt complet, appeler "create_dataloader_v1".
# Utiliser batch_size=1, max_length=4, stride=1 et shuffle=False pour simplifier.
# Ensuite, exécuter "iter(dataloader)" et récupérer le prochain batch avec "next()"
# Observer l'affichage des deux premiers tenseurs des IDs.

In [52]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

dataloader = create_dataloader_v1(
raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)

# Converts dataloader into a Python iterator to fetch the next entry via Python's built-in next() function
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [53]:
# Appeler une deuxième fois "next(data_iter)" 
# Cette étape permet d'observer la gestion du décalage (stride=1) 
# qui fait glisser la fenêtre d'une position pour ce second lot.

In [54]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [55]:
# Créer cette fois-ci "dataloader" avec batch_size=8, max_length=4 et stride=4, ainsi que shuffle=False
# Transformer en itérateur et appeler next()
# Assigner aux variables "inputs" et "targets" puis les afficher.
# Remarquer comment l'augmentation du paramètre "stride" empêche dorénavant les superpositions.

In [56]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
data_iter = iter(dataloader)
inputs, target = next(data_iter)

print("Inputs:\n", inputs)
print("Targets:\n", target)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


### Exercice 2.2 Data loaders with different strides and context sizes

In [57]:
# Tester le chargement des données avec une nouvelle configuration : max_length=2 et stride=2.
# Puis avec : max_length=8 et stride=2. Observer les résultats.

In [58]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=2, stride=2, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(f"first batch: {first_batch}")
second_batch = next(data_iter)
print(f"second batch: {second_batch}")

first batch: [tensor([[ 40, 367]]), tensor([[ 367, 2885]])]
second batch: [tensor([[2885, 1464]]), tensor([[1464, 1807]])]


In [59]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=8, stride=2, shuffle=False)
data_iter = iter(dataloader)
first_batch = next(data_iter)
print(f"first batch: {first_batch}")
second_batch = next(data_iter)
print(f"second batch: {second_batch}")

# le stride est la position dans le tensor à partir de laquelle deux entrées successives se chevauchent

first batch: [tensor([[  40,  367, 2885, 1464, 1807, 3619,  402,  271]]), tensor([[  367,  2885,  1464,  1807,  3619,   402,   271, 10899]])]
second batch: [tensor([[ 2885,  1464,  1807,  3619,   402,   271, 10899,  2138]]), tensor([[ 1464,  1807,  3619,   402,   271, 10899,  2138,   257]])]


### 2.7 Comprendre la différence entre les couches d’embedding et les couches linéaires

* Les couches d’embedding dans **PyTorch** accomplissent la même chose que des couches linéaires qui effectuent des multiplications matricielles ; la raison pour laquelle on utilise des couches d’embedding est l’efficacité computationnelle.

* Nous allons examiner cette relation étape par étape à l’aide de code et d’exemples en **PyTorch**.


#### 2.7.1 Utiliser `nn.Embedding`

In [60]:
# Supposons que nous avons les 3 instances d'entrainement suivant
# qui représente les tokens IDs dans le contexte d'un LLM
idx = torch.tensor([2, 3, 1])

# le nombre de lignes dans la matrice d'embedding peut être déterminé
# en trouvant le token ID le plus élevé + 1 
# Si le token ID le plus élevé est 3, alors nous avons 4 lignes, soient 
# les tokens IDS 0, 1, 2, 3
num_idx = max(idx) + 1

# La dimension désirée de l'embedding est un hyperparamètre
out_dim = 5

* Implémentons une simple couche d'embedding

In [61]:
# Nous utilisons le random seed pour la reproductibilité puisque
# les poids de l'embedding sont initialisés aléatoirement
torch.manual_seed(123)

embedding = torch.nn.Embedding(num_idx, out_dim)

Pour voir les poids de la matrice d'embedding

In [62]:
embedding.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  1.5810],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015],
        [ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953]], requires_grad=True)

* On peut alors utiliser les couches d'embedding pour obtenir la représentation vectorielle de l'instance d'entrainement d'ID 1:

In [63]:
embedding(torch.tensor([1]))

tensor([[ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

* Ci-dessous se trouve la visualisation de ce qui passe en arrière-plan :

<div align="center">
    <img src="img/vis_1.png">
</div>

* De façon similaire, nous pouvons utiliser la matrice d'embedding pour obtenir une représentation vectorielle de l'instance d'entrainement d'ID 2:

In [64]:
embedding(torch.tensor([2]))

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315]],
       grad_fn=<EmbeddingBackward0>)

<div align="center">
    <img src="img/vis_2.png">
</div>

* À présent, convertissons toutes les instances d'entrainement que nous avions défini précédemment. 

In [65]:
idx = torch.tensor([2, 3, 1])
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

* D'un point de vue technique, la logique de table de correspondance reste inchangée.

<div align="center">
    <img src="img/vis_3.png">
</div>

#### 2.7.1 Utiliser `nn.Linear`

* À présent, nous allons démontrer que la couche d'embedding ci-dessus réalise exactement la même opération qu'une couche `nn.Linear` appliquée à une représentation encodée en one-hot dans Pytorch.


* Commençons d'abord par convertir les identifiants de tokens en une représentation one-hot :

In [66]:
onehot = torch.nn.functional.one_hot(idx)
onehot

tensor([[0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]])

* Initialisons ensuite la couche `Linear` qui effecture une multiplication matricielle **$XW^{T}$**

In [67]:
torch.manual_seed(123)
linear = torch.nn.Linear(num_idx, out_dim, bias=False)
linear.weight

Parameter containing:
tensor([[-0.2039,  0.0166, -0.2483,  0.1886],
        [-0.4260,  0.3665, -0.3634, -0.3975],
        [-0.3159,  0.2264, -0.1847,  0.1871],
        [-0.4244, -0.3034, -0.1836, -0.0983],
        [-0.3814,  0.3274, -0.1179,  0.1605]], requires_grad=True)

* Notons que la couche linéaire dans Pytorch est également initialisée avec de petits poids aléatoires: afin de la comparer directement à la couche `Embedding` ci-dessus, nous devons utiliser les mêmes petits poids aléatoires, c'est pourquoi nous les réattribuons ici :

In [68]:
linear.weight = torch.nn.Parameter(embedding.weight.T)

---

La transposée `.T` ici sert à **anticiper le calcul interne de la couche `nn.Linear`** pour que la multiplication matricielle avec la matrice one-hot se déroule correctement.

En effet, le but est de faire une multiplication matricielle standard entre notre entrée et nos poids d'embedding :
* `one_hot_matrix` a pour shape : `(batch_size, vocab_size)`
* `embedding.weight` a pour shape : `(vocab_size, embedding_dim)`

Mathématiquement, le produit matriciel naturel $X \cdot W_{emb}$ fonctionnerait parfaitement :
`(batch_size, vocab_size) × (vocab_size, embedding_dim) = (batch_size, embedding_dim)`

**Le problème avec `nn.Linear` :**
En PyTorch, la couche `nn.Linear(in_features, out_features)` avec des poids $W_{lin}$ n'effectue pas une simple multiplication $X \cdot W_{lin}$. Sous le capot, elle calcule toujours **$X \cdot W_{lin}^T$** (la matrice d'entrée multipliée par la **transposée** de ses propres poids).

* Si l'on assignait directement `linear.weight = embedding.weight`, PyTorch calculerait lors du passage en avant : $X \cdot (embedding.weight)^T$.
* L'opération deviendrait alors : `(batch_size, vocab_size) × (embedding_dim, vocab_size)`, ce qui provoquerait une **erreur de dimension**.

**La solution :**
Pour contourner cela, on donne à la couche Linéaire la **transposée** de nos poids d'embedding :
```python
linear.weight = torch.nn.Parameter(embedding.weight.T)
```

Ainsi, lorsque `Linear` applique sa formule interne $X \cdot W_{lin}^T$, le calcul devient :
$X \cdot (W_{emb}^T)^T  =  X \cdot W_{emb}$

**En résumé :** On transpose les poids à l'assignation parce que `nn.Linear` va les transposer à nouveau lors du calcul. Les deux transposées s'annulent ($(A^T)^T = A$), ce qui permet de retrouver exactement la multiplication matricielle voulue entre la matrice one-hot et les poids d'origine.

---

* Nous pouvons maintenant utiliser la couche linéaire sur la représentation one-hot des entrées :

In [69]:
linear(onehot.float())

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]], grad_fn=<MmBackward0>)

* Comme nous pouvons le voir, c'est exactement la même chose que quand nous avons utilisé la couche d'embedding

In [70]:
embedding(idx)

tensor([[ 0.6957, -1.8061, -1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096, -0.4076,  0.7953],
        [ 1.3010,  1.2753, -0.2010, -0.1606, -0.4015]],
       grad_fn=<EmbeddingBackward0>)

* Ce qui se passe en interne correspond à la computation suivante pour l'identifiant de token du premier exemple d'entrainement :

<div align="center">
    <img src="img/vis_4.png">
</div>


* Et pour l'identifiant de la  seconde instance d'entrainement :

<div align="center">
    <img src="img/vis_5.png">
</div>

* Puisque tous les indices d’une ligne one-hot, sauf un seul, sont égaux à 0 par construction, cette multiplication matricielle revient essentiellement à faire une simple recherche des éléments correspondants au vecteur one-hot.

* L’utilisation de la multiplication matricielle sur des encodages one-hot est donc équivalente à la consultation d’une couche d’embedding, mais elle peut être inefficace lorsque l’on travaille avec de grandes matrices d’embedding, car elle entraîne beaucoup de multiplications inutiles par zéro.


### 2.8 Encoding word positions

##### 2.8.1 Initialisation de l'embedding des mots

In [71]:
# Définition de la taille du vocabulaire (ici adapté au BPE tokenizer)
vocab_size = 50257
# Définition de la dimension du vecteur de sortie pour chaque token
output_dim = 5
# Création de la couche d'embedding pour les tokens 
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

##### 2.8.2 Préparation du Dataloader et extraction d'un batch

In [72]:
# On fixe la longueur maximale de la séquence à 4 tokens
max_length = 4

# instanciation du DataLoader avec les paramètres configurés précédemment
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=max_length,
    stride=max_length, shuffle=False
)

# Création d'un itérateur et extraction du premier lot (batch) d'entrées et cibles
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

print("Tokens IDs:\n", inputs)
print("\nForme des entrées (Input shape):\n",inputs.shape)

Tokens IDs:
 tensor([[  40,  367, 2885, 1464]])

Forme des entrées (Input shape):
 torch.Size([1, 4])


##### 2.8.3 Passage des entrées dans l'embedding des mots

In [73]:
# On intègre les IDs des tokens d'entrée dans nos vecteurs à 256 dimensions
token_embeddings = token_embedding_layer(inputs)

# Afficher la forme du nouveau tenseur (batch_size, seq_len, output_dim)
print(token_embeddings.shape)

torch.Size([1, 4, 5])


##### 2.8.4 Création des embeddings de position absolue

In [74]:
# la longueur de contexte est similaire à la longueur maximale du texte en entrée
context_length = max_length

# Création de la couche d'embedding pour les positions
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# On génère les vecterurs de position en passant un vecteur allant de 0 à context_length - 1 
pos_embeddings = pos_embedding_layer(torch.arange(context_length))

# Afficher la forme des embeddings de position
print(pos_embeddings.shape)

torch.Size([4, 5])


##### 2.8.5 L'injection de la position

In [75]:
# On additionne les embeddings de tokens initiaux avec les embeddings de position 
# Pytorch utilise le mécanisme de broadcasting pour appliquer les positions à chaque exemple du batch
input_embeddings = token_embeddings + pos_embeddings

# Afficher la formme finale
print(input_embeddings.shape)

torch.Size([1, 4, 5])


#### `NB` :

##### - `Longueur de contexte vs longueur maximale du texte`

Ce sont **la même contrainte**, vue depuis deux angles différents.

`context_length` définit la taille de la matrice positionnelle : `(context_length, output_dim)`. C'est donc le **plafond absolu** du modèle — aucune séquence ne peut dépasser ce nombre de tokens, sinon `IndexError`.

`max_length` dans le DataLoader, c'est la longueur des séquences que vous découpez pour l'entraînement. Dans le livre, on pose `context_length = max_length` simplement pour que les deux soient cohérents. En pratique, `max_length` doit toujours être ≤ `context_length`.

---

##### - `Broadcasting`

Les deux tenseurs à additionner ont des formes différentes :

```
token_embeddings  →  (8, 4, 256)   # 8 exemples, 4 tokens, 256 dims
pos_embeddings    →     (4, 256)   # 4 positions, 256 dims
```

PyTorch ne peut pas additionner directement des tenseurs de formes différentes. Le *broadcasting* résout ça : il **répète automatiquement** `pos_embeddings` 8 fois en mémoire pour lui donner la forme `(8, 4, 256)`, puis effectue l'addition.

Concrètement : les mêmes vecteurs de position 0, 1, 2, 3 sont appliqués à chacun des 8 exemples du batch — ce qui est exactement ce qu'on veut, puisque la notion de « position 0 » est la même pour toutes les phrases.